In [ ]:
#-------------------------------------------------------------------------------
# Name:        03_convert_Masks
# Purpose:     Rasterize the vecor input mask so that they are 128x128px
#
# Author:      Gijs van den Dool
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/1dgFGkQ0Orjt8SbPtpOvPNb1A9y4Odeng?usp=drive_link

# Vector Masks:
# Output from the selection of ISL areas in a 1280x1280 bounding box

# Raster Masks:
# Input for the modelling, as a direct conversion from vector to raster


In [ ]:
# check packages installed:
packages = !pip list

if "mapclassify" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install mapclassify -q


if "rasterio" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install rasterio -q

In [ ]:
import os

import geemap

import json

from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import geopandas as gpd

from matplotlib import pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_ISL"
task_folder = "02_WorkingFiles"
file_name = "mask_ISL_02.geojson" # might be different for other areas

file_path = os.path.join(project_root, task_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File exists.")
else:
    print("File does not exist or is not accessible.")


File exists.


In [ ]:
# Create output folder:
task_folder = "03_Output_Data"
out_folder = "01_masks/"
out_folder = os.path.join(project_root, task_folder, out_folder)

if not os.path.exists(out_folder):
    os.makedirs(out_folder)
    print("Folder created successfully.")
else:
    print("Folder already exists.")


Folder created successfully.


In [ ]:
gdf_ISL_Mask = gpd.read_file(file_path)

print(len(gdf_ISL_Mask)) # total lines in the AoI
print(gdf_ISL_Mask.crs)  # current projection
gdf_ISL_Mask.head(3)

36
EPSG:4326


,GID,AoI_id_2,mask,geometry
0,GID_ISL_02_0_3,ISL_02,1,"POLYGON ((15.11921 2.27183, 15.11922 2.27184, ..."
1,GID_ISL_02_1_0,ISL_02,1,"POLYGON ((15.13593 2.30917, 15.13593 2.30918, ..."
2,GID_ISL_02_1_2,ISL_02,1,"MULTIPOLYGON (((15.13656 2.28529, 15.13653 2.2..."


In [ ]:
#create an unique list of masks:
lst_GID = gdf_ISL_Mask.GID.unique().tolist()

In [ ]:
# pixels by patch: ISL:
shape = 128, 128

In [ ]:
# ToDo: add CRS to the exported tiff
# crs="+proj=latlong",

# creating the masks:
for i in range(len(lst_GID)):
  str_image_name = lst_GID[i] + '.tif'
  image_path = out_folder + str_image_name

  # set the patch extent
  rslt_df= gdf_ISL_Mask.loc[(gdf_ISL_Mask['GID'] == lst_GID[i])]
  rslt_df

  # set the mask
  mask_df = gdf_ISL_Mask.loc[(gdf_ISL_Mask['GID'] == lst_GID[i]) & (gdf_ISL_Mask['mask'] == 1) ]
  mask_df

  # The projection is kept in the original projection, but here it is possible to
  # change if an other projection is needed:
  # rslt_df = rslt_df.to_crs("32632")
  # rslt_df.crs

  # mask_df = mask_df.to_crs("32632")
  # mask_df.crs

  # set the transformation:
  transform = rasterio.transform.from_bounds(*rslt_df['geometry'].total_bounds, *shape)
  transform

  # create the mask:
  rasterize_mask = rasterize(
      [(shape, 1) for shape in mask_df['geometry']],
      out_shape=shape,
      transform=transform,
      fill=0,
      all_touched=True,
      dtype=rasterio.uint8,
      default_value=1)

  # save the raster:
  with rasterio.open(
      image_path, 'w',
      driver='GTiff',
      dtype=rasterio.uint8,
      count=1,
      width=shape[0],
      height=shape[1],
      transform=transform
  ) as dst:
      dst.write(rasterize_mask, indexes=1)

In [ ]:
image_list = os.listdir('/Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR')
mask_list = os.listdir('/Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_ISL_057/03_Output_Data/01_masks')

print(mask_list)
print(image_list)

['GID_ISL_02_3_0.tif', 'GID_ISL_02_1_2.tif', 'GID_ISL_02_1_3.tif', 'GID_ISL_02_3_1.tif', 'GID_ISL_02_3_3.tif', 'GID_ISL_02_1_0.tif', 'GID_ISL_02_3_2.tif', 'GID_ISL_02_5_1.tif', 'GID_ISL_02_5_2.tif', 'GID_ISL_02_2_3.tif', 'GID_ISL_02_2_2.tif', 'GID_ISL_02_2_0.tif', 'GID_ISL_02_2_1.tif', 'GID_ISL_02_0_3.tif', 'GID_ISL_02_4_3.tif', 'GID_ISL_02_4_2.tif', 'GID_ISL_02_4_0.tif', 'GID_ISL_02_4_1.tif']
['image_GID_ISL_02_3_3.tif', 'image_GID_ISL_02_3_2.tif', 'image_GID_ISL_02_1_0.tif', 'image_GID_ISL_02_1_2.tif', 'image_GID_ISL_02_3_0.tif', 'image_GID_ISL_02_3_1.tif', 'image_GID_ISL_02_1_3.tif', 'image_GID_ISL_02_5_2.tif', 'image_GID_ISL_02_5_1.tif', 'image_GID_ISL_02_2_0.tif', 'image_GID_ISL_02_0_3.tif', 'image_GID_ISL_02_2_1.tif', 'image_GID_ISL_02_2_3.tif', 'image_GID_ISL_02_2_2.tif', 'image_GID_ISL_02_4_0.tif', 'image_GID_ISL_02_4_1.tif', 'image_GID_ISL_02_4_3.tif', 'image_GID_ISL_02_4_2.tif', 'image_GID_ISL_02_3_1.zip']


Affine(10.000030862003769, 0.0, 1390778.0,
       0.0, -10.00000000000091, 372062.00000000006)

In [ ]:
[(shape, 1) for shape in mask_df['geometry']]

[(<POLYGON ((1392053.665 371642.821, 1391921.766 371414.653, 1391920.169 37141...>,
  1)]